# Extract and standardize data from the 12 validated monthly CSV files
# Define the locked manifest of 54 variables
# Read each monthly CSV using pandas.read_csv using usecols to restrict to the manifest
# Explicitly set dtpes for critical fields
# Save each monthly extract as a cleaned CSV

In [ ]:
from pathlib import Path
import pandas as pd

# Locked manifest (authoritative list of 55 variables)
FINAL_MANIFEST = [ 
"N_AIH", 
"IDADE", 
"COD_IDADE",  
"SEXO",
"RACA_COR", 
"NACIONAL", 
"ETNIA",
"INSTRU",
"CBOR",
"MUNIC_RES", 
"MUNIC_MOV",
"DIAG_PRINC",
"DIAGSEC1",
"DIAGSEC2",
"DIAGSEC3",
"DIAGSEC4",
"DIAGSEC5",
"DIAGSEC6",
"DIAGSEC7",
"DIAGSEC8",
"DIAGSEC9",
"TPDISEC1",
"TPDISEC2",
"TPDISEC3",
"TPDISEC4",
"TPDISEC5",
"TPDISEC6",
"TPDISEC7",
"TPDISEC8",
"TPDISEC9",
"PROC_SOLIC",
"PROC_REA",
"MORTE",
"DIAS_PERM",
"ANO_CMPT",
"MES_CMPT",
"DT_INTER",
"DT_SAIDA",
"MARCA_UTI",
"MARCA_UCI",
"QT_DIARIAS",
"UTI_MES_TO",
"UTI_INT_TO",
"CNES",
"ESPEC",
"NAT_JUR",
"GESTAO",
"VAL_TOT",
"VAL_SH",
"VAL_SP",
"VAL_UTI",
"VAL_UCI",
"FINANC",
"FAEC_TP"
]

dtype_map = {
    "N_AIH" : "int64",
    "IDADE" : "int64",
    "COD_IDADE" : "int64",
    "SEXO" : "int64",
    "RACA_COR"  : "int64",
    "NACIONAL"  : "int64",
    "ETNIA": "string",  
    "INSTRU": "int64",
    "CBOR": "int64",
    "MUNIC_RES": "int64",
    "MUNIC_MOV": "int64",
    "DIAG_PRINC": "string",
    "DIAG_SECUN": "int64",
    "DIAGSEC1": "string",
    "DIAGSEC2": "string",
    "DIAGSEC3": "string",
    "DIAGSEC4": "string",
    "DIAGSEC5": "string",
    "DIAGSEC6": "string",
    "DIAGSEC7": "string",
    "DIAGSEC8": "string",
    "DIAGSEC9": "string",
    "TPDISEC1": "int64",
    "TPDISEC2": "int64",
    "TPDISEC3": "int64",
    "TPDISEC4": "int64",
    "TPDISEC5": "int64",
    "TPDISEC6": "int64",
    "TPDISEC7": "int64",
    "TPDISEC8": "int64",
    "TPDISEC9": "int64",
    "PROC_SOLIC": "int64",
    "PROC_REA": "int64",
    "MORTE": "int64",
    "DIAS_PERM": "int64",
    "ANO_CMPT": "int64",
    "MES_CMPT": "int64",
    "DT_INTER": "int64",
    "DT_SAIDA": "int64",
    "MARCA_UTI": "int64",
    "MARCA_UCI": "int64",
    "QT_DIARIAS": "int64",
    "UTI_MES_TO": "int64",
    "UTI_INT_TO": "int64",
    "CNES": "int64",
    "ESPEC": "int64",
    "NAT_JUR": "int64",
    "GESTAO": "int64",
    "VAL_TOT": "float64",
    "VAL_SH": "float64",
    "VAL_SP": "float64",
    "VAL_UTI": "float64",
    "VAL_UCI": "float64",
    "FINANC": "int64",
    "FAEC_TP": "string"
}

# Define folder paths as Path onjects
raw_dir = Path("/raw_data")
out_dir = Path("/data_extraction")


def extract_monthly(file_path):
    return pd.read_csv(file_path, sep=';', encoding='latin-1', usecols=FINAL_MANIFEST, dtype=dtype_map)

# Path.glob() matches files directly inside the dirctory
for file_path in sorted(raw_dir.glob("RD2024*.csv")):
    df = extract_monthly(file_path)

    # Safely construct destination path without string manipulation bugs
    out_name = out_dir/file_path.name

    df.to_csv(out_name, index=False)
    print("Saving to:", out_name)


# Extracted files sanity check loop

In [ ]:
import pandas as pd
from pathlib import Path

FINAL_MANIFEST = [
    "N_AIH", "IDADE", "COD_IDADE", "SEXO", "RACA_COR", "NACIONAL",
    "ETNIA", "INSTRU", "CBOR", "MUNIC_RES", "MUNIC_MOV",
    "DIAG_PRINC", 
    "DIAGSEC1", "DIAGSEC2", "DIAGSEC3", "DIAGSEC4", "DIAGSEC5",
    "DIAGSEC6", "DIAGSEC7", "DIAGSEC8", "DIAGSEC9",
    "TPDISEC1", "TPDISEC2", "TPDISEC3", "TPDISEC4", "TPDISEC5",
    "TPDISEC6", "TPDISEC7", "TPDISEC8", "TPDISEC9",
    "PROC_SOLIC", "PROC_REA", "MORTE", "DIAS_PERM",
    "ANO_CMPT", "MES_CMPT", "DT_INTER", "DT_SAIDA",
    "MARCA_UTI", "MARCA_UCI", "QT_DIARIAS", "UTI_MES_TO", "UTI_INT_TO",
    "CNES", "ESPEC", "NAT_JUR", "GESTAO",
    "VAL_TOT", "VAL_SH", "VAL_SP", "VAL_UTI", "VAL_UCI",
    "FINANC", "FAEC_TP"
]

out_dir = Path("/data_extraction")

summary = []

for file_path in sorted(out_dir.glob("RD2024*.csv")):

    print("\n--- Checking:", file_path.name, "---")

    # Read extracted file
    df = pd.read_csv(
        file_path,
        sep=",",
        encoding="utf-8"
    )

    # Basic dimensions
    rows = df.shape[0]
    columns = df.shape[1]

    # Dtype summary
    dtype_counts = df.dtypes.astype(str).value_counts().to_dict()

    # Missingness
    missing = df.isnull().sum()
    missing = missing[missing > 0]

    # Column check
    missing_cols = set(FINAL_MANIFEST) - set(df.columns)
    extra_cols = set(df.columns) - set(FINAL_MANIFEST)

    # Build one summary row for this month
    row = {
        "file": file_path.name,
        "rows": rows,
        "columns": columns
    }

    row.update(dtype_counts)

    row.update({
        "columns_with_missing": len(missing),
        "total_missing_cells": missing.sum(),
        "missing_columns": list(missing_cols),
        "extra_columns": list(extra_cols)
    })

    summary.append(row)


# Create summary AFTER all 12 files have been processed
extraction_summary = pd.DataFrame(summary)

print("\n--- Extraction Summary ---")
print(extraction_summary)